In [ ]:
!pip install --upgrade numpy pandas

## ISOLATION FOREST (FIRST ML MODEL)

In [ ]:
from sqlalchemy import create_engine, text

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("numpy:", np.__version__)    # should print 1.26.4
print("All good!")

In [ ]:
class FeatureEngineer:
    def __init__(self, db_url: str):
        self.engine = create_engine(db_url)

    def load_ohlcv(self, ticker: str, lookback: int = 500) -> pd.DataFrame:
        query = """
            Select ts, open, high, low, close, volume,
            From market_data where ticker = :ticker
            ORDER BY ts DESC LIMIT :n
        """
        with self.engine.connect() as conn:
            df = pd.read_sql(text(query), conn,
                             params={"ticker":ticker, "n": lookback})
            return df.sort_values("ts").reset_index(drop="True")

    def compute (self, df: pd.DataFrame) -> pd.DataFrame:
        #Log returns
        df["log_return"]=np.log(df["close"] / df["close"].shift(1))
        #RSI-14
        delta = df["close"].diff()
        gain = delta.clip(lower=0).ewm(span=14).mean()
        loss = (-delta).clip(lower=0).ewm(span=14).mean()
        df["rsi_14"] = 100 - (100 / (1 + gain / loss))
        #Bollinger Bands (20, 2σ)
        ma = df["close"].rolling(20).mean()
        std = df["close"].rolling(20).std()
        df["bb_upper"] = ma + 2 * std
        df["bb_lower"] = ma - 2 * std
        #VWAP
        df["vwap"] = (df["close"] * df["volume"]).cumsum() / df["volume"].cumsum()
        #Volume ratio (vs 20-bar avg)
        df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean()
        return df.dropna()

In [ ]:
pip install yfinance sqlalchemy pymysql pandas

In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/Users/Aarushi/.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected! (password hidden safely)")

In [ ]:
import yfinance as yf
import pandas as pd

#These are the stocks we want to track
TICKERS = ["AAPL", "SPY", "TSLA"]

for ticker in TICKERS:
    print(f"Downloading {ticker}...")

    #Download 5 years of daily prices
    df = yf.download(ticker, period = "5y", interval = "1d", auto_adjust = True, progress = False)

    #Tidy up the columns
    df = df.reset_index()
    df.columns = ["ts", "open", "high", "low", "close", "volume"]
    df["ticker"] = ticker

    #Save to MySQL (if the row already exists, skip it)
    df.to_sql("market_data", con = engine, if_exists = "append", index = False)

    print(f"  Saved {len(df)} rows for {ticker}")
print("Done! All data is in MySQL.")

In [ ]:
print(engine.url)

In [ ]:
import pandas as pd

query = """
SELECT 
    ticker, 
    COUNT(*) AS total_rows, 
    MIN(ts) AS earliest, 
    MAX(ts) AS latest
FROM market_data
GROUP BY ticker
"""

result = pd.read_sql(query, engine)
print(result)

#### IMPORTING TOOLS

In [ ]:
import pandas as pd                               #for working with tables of data
import numpy as np                                #for maths
import matplotlib.pyplot as plt                   #for drawing charts/graphs
import seaborn as sns                             #for prettier charts
from sqlalchemy import create_engine, text        #for talking to MySQL
from sklearn.ensemble import IsolationForest      #the main model
from sklearn.preprocessing import RobustScaler    #for scaling numbers
import warnings
warnings.filterwarnings("ignore")                 #hides unneccesary warning messages

print ("All tools imported successfully!")

#### CONNECT TO MySQL

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/Users/Aarushi/.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected! (password hidden safely)")

#### LOAD DATA FROM SQL

In [ ]:
TICKER = "AAPL"

query = f"""
    SELECT ts, open, high, low, close, volume
    FROM market_data
    WHERE ticker = '{TICKER}'
    ORDER BY ts ASC
"""

df = pd.read_sql(query, engine, parse_dates=["ts"])

print (f"Loaded {len(df)} rows for {TICKER}")
print (df.head())

#### FEATURE ENGINEERING

In [ ]:
def compute_features(df):
    """
    Takes a table of raw stock prices.
    Returns the same table with 6 new feature columns added.
    """
    df = df.copy()   # Make a copy so we don't mess up the original
    df = df.sort_values("ts").reset_index(drop=True)
 
    #── Feature 1: Log Return ─────────────────────────────────
    #How much did the price change compared to yesterday?
    #Example: if price went from 100 to 102, log return ≈ 0.02
    #Big log returns (positive OR negative) = unusual price move
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
 
    #── Feature 2: RSI (Relative Strength Index) ──────────────
    #RSI tells us if a stock is being overbought or oversold.
    #Think of it like a thermometer for "excitement" about a stock.
    #RSI near 100 = everyone is buying like crazy (overheated)
    #RSI near 0   = everyone is selling like crazy (panic)
    #Both extremes can be anomalies!
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).ewm(span=14, min_periods=14).mean()
    loss  = (-delta).clip(lower=0).ewm(span=14, min_periods=14).mean()
    rs    = gain / (loss + 1e-9)   # 1e-9 prevents dividing by zero
    df["rsi_14"] = 100 - (100 / (1 + rs))
 
    #── Feature 3 & 4: Bollinger Band Distances ───────────────
    #Bollinger Bands are like a "normal range lane" for prices.
    #Imagine the price usually drives between two white lines on a road.
    #If it suddenly swerves way outside those lines : anomaly!
    
    #bb_upper_dist: how far ABOVE the upper line is the price?
    #bb_lower_dist: how far BELOW the lower line is the price?
    ma  = df["close"].rolling(20).mean()   # 20-day average price
    std = df["close"].rolling(20).std()    # How much prices vary
    bb_upper = ma + 2 * std
    bb_lower = ma - 2 * std
    df["bb_upper_dist"] = (bb_upper - df["close"]) / (df["close"] + 1e-9)
    df["bb_lower_dist"] = (df["close"] - bb_lower) / (df["close"] + 1e-9)
 
    #── Feature 5: VWAP Distance ──────────────────────────────
    #VWAP = Volume Weighted Average Price
    #It's the "fair value" of the stock based on both price AND volume.
    #Professional traders use VWAP as a baseline.
    #A price far from VWAP = something unusual is happening.
    typical_price = (df["high"] + df["low"] + df["close"]) / 3
    cumulative_tp_vol = (typical_price * df["volume"]).cumsum()
    cumulative_vol    = df["volume"].cumsum()
    vwap = cumulative_tp_vol / (cumulative_vol + 1e-9)
    df["vwap_dist"] = (df["close"] - vwap) / (vwap + 1e-9)
 
    #── Feature 6: Volume Ratio ───────────────────────────────
    #On a normal day, AAPL might trade 50 million shares.
    #On a crazy news day, it might trade 500 million shares.
    #Volume ratio = today's volume / average volume over last 20 days
    #A ratio of 10 means: 10x more volume than usual : suspicious!
    df["vol_ratio"] = df["volume"] / (df["volume"].rolling(20).mean() + 1e-9)
 
    #Drop rows where we couldn't compute features (early rows have NaN)
    df = df.dropna().reset_index(drop=True)
 
    return df
 
 
#Run the function on our data
df = compute_features(df)
 
print(f"Features computed! Shape: {df.shape}")
print(df[["ts", "close", "log_return", "rsi_14", "vol_ratio"]].tail(10))
 

#### SCALING THE FEATURES

In [ ]:
FEATURE_COLS = [
    "log_return",
    "rsi_14",
    "bb_upper_dist",
    "bb_lower_dist",
    "vwap_dist",
    "vol_ratio",
]

scaler = RobustScaler()

X_scaled = scaler.fit_transform(df[FEATURE_COLS])

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print("(rows = trading days, columns = 6 features)")

#### TRAIN THE ISOLATION FOREST MODEL

In [ ]:
#This is where the magic happens

#We are creating the model and teaching it what normal looks like

#Key settings explained:
#n_estimators = 200    Build 200 "decision trees". (More = more accurate)
#contamination = 0.02  We're telling it: "expect about 2% of days to be anomalies". This is a good starting guess
#random_state = 42     Makes results reproducible (same answer every run)
#n_jobs = -1           Use all CPU cores (faster training)

model = IsolationForest(
    n_estimators = 200,
    contamination = 0.02,
    max_features = 0.8,
    random_state = 42,
    n_jobs = -1,
)

#.fit() = "study this data and learn what normal looks like"
model.fit(X_scaled)

print("Model trained!")
print(f"It studied {len(X_scaled)} trading days of {TICKER} data.")

#### GET ANOMALY SCORES

In [ ]:
#Now we ask the model -- "For each day, how weird was it"?

#score_samples() returns a negative number for each row
#Very negative = Very anomalous (unusual)
#Close to 0 = normal

#We then FLIP and NORMALISE these scores to a 0-1 range
#where 1.0 = most anomalous and 0.0 = completely normal
#This makes it easier to understand and compare

#Get raw scores (negative numbers)
raw_scores = model.score_samples(X_scaled)

#Flip (multiply by -1) and normalise to a 0-1 range
#Formula : (value - min) / ((max - min)
anomaly_scores = (-raw_scores - (-raw_scores).min()) / \
                 ((-raw_scores).max() - (-raw_scores).min())

#Add scores to our dataframe
df["iso_score"] = anomaly_scores
df["is_anomaly"] = (anomaly_scores >= 0.65).astype(int)
#is_anomaly = 1 means "flagged as anomaly", 0 means "normal"

print(f"Total trading days analysed: {len(df)}")
print(f"Days flagged as anomalies: {df['is_anomaly'].sum()}")
print(f"Anomaly rate: {df['is_anomaly'].mean():.1%}")

#### LOOK AT THE TOP ANOMALIES

In [ ]:
#Lets actually see which days the model flagged
#We sort by anomaly score (highest = most suspicious)
#Show the top 15 most unusual trading days

top_anomalies = (
    df[df["is_anomaly"]==1]
    .sort_values("iso_score", ascending = False)
    [["ts", "close", "log_return", "vol_ratio", "rsi_14", "iso_score"]]
    .head(15)
    .reset_index(drop=True)
)

print(f"\nTop 15 anomalous trading days for {TICKER}:")
print("=" * 70)
print(top_anomalies.to_string(index = False))

#### ANOMALY SCORE OVER TIME

In [ ]:
#This chart shows:
# - The stock's closing price as a line
# - Red dots on top wherever the model flagged an anomaly
#A good model will put red dots on dates you recognize as newsworthy (crashes, pumps, earnings beats/misses).

fig, (ax1, ax2) = plt.subplots(2,1, figsize = (14,8), sharex = True)
fig.suptitle (f"{TICKER} - Isolation Forest Anomaly Detection", fontsize = 14)

#Top chart : price line + anomaly dots
ax1.plot(df["ts"], df["close"], color = "#378ADD", linewidth = 1, label = "Close price")
anomaly_mask = df["is_anomaly"] == 1
ax1.scatter (
    df.loc[anomaly_mask, "ts"],
    df.loc[anomaly_mask, "close"],
    color = "#D85A30", s = 40, zorder = 5, label = "Anomaly flagged"
)
ax1.set_ylabel("Price ($)")
ax1.legend(loc = "upper left")
ax1.set_title("Stock price with anomalies highlighted")

#Bottom chart: the anomaly score itself over time
ax2.fill_between(df["ts"], df["iso_score"], color = "#534Ab7", linewidth = 0.8)
ax2.axhline(y=0.65, color = "#D85A30", linestyle = "--", linewidth = 1,
            label = "Threshold (0.65)")
ax2.set_ylabel("Anomaly score (0-1)")
ax2.set_xlabel ("Date")
ax2.legend(loc="upper left")
ax2.set_title("anomaly score over time (above red line = flagged)")

plt.tight_layout()
plt.savefig(f"{TICKER}_iso_anomalies.png", dpi = 150, bbox_inches = "tight")
plt.show()
print(f"Chart saved as {TICKER}_iso_anomalies.png")

#### SCORE DISTRIBUTION HISTOGRAM

In [ ]:
#This histogram answers: "Are most days normal, with just a few weird ones?" - which is what we want to see
#A healthy distribution has:
# - A big peak on the LEFT (lots of normal days)
# - A small tail on the RIGHT ( a few anomalous days)
#The black vertical line is our threshold

fig, ax = plt.subplots(figsize=(10,4))

ax.hist(
    df.loc[df["is_anomaly"] == 0, "iso_score"],
    bins = 50, color = "#378ADD", alpha = 0.7, label = "Normal days", density = True
)
ax.hist(
    df.loc[df["is_anomaly"] == 1, "iso_score"],
    bins = 20, color = "#D85A30", alpha = 0.7, label = "Anomalous days", density = True
)
ax.axvline(x=0.65, color = "black", linestyle = "--", linewidth = 1.5,
           label = "Threshold (0.65)")

ax.set_xlabel("Anomaly score (0 = normal, 1 = Very anomalous)")
ax.set_ylabel("Density")
ax.set_title(f"{TICKER} - Distribution of Anomaly Scores")          
ax.legend()

plt.tight_layout()
plt.savefig(f"{TICKER}_iso_distribution.png", dpi = 150, bbox_inches = "tight")
plt.show()

#### FEATURE HEATMAP FOR TOP ANOMALIES

In [ ]:
#This shows why each anomaly was flagged
#Each row = one anomalous day
#Each column = one feature
#Dark red = Very high value (unusual), dark blue = very low

top20 = df[df["is_anomaly"] == 1].sort_values("iso_score", ascending = False).head(20)
heat_data = top20[FEATURE_COLS].copy()
heat_data.index = top20["ts"].dt.strftime("%Y-%m-%d")

fig, ax = plt.subplots(figsize = (10,8))
sns.heatmap(
    heat_data,
    cmap = "RdBu_r",
    center = 0,
    annot = True,
    fmt = ".2f",
    linewidth = 0.3,
    ax=ax,
)
ax.set_title = (f"{TICKER} - Feature values for top 20 anomalous days")
ax.set_xlabel("Feature")
ax.set_ylabel("Date")

plt.tight_layout()
plt.savefig(f"{TICKER}_iso_heatmap.png", dpi = 150, bbox_inches = "tight")
plt.show()

#### SAVE RESULTS TO MYSQL

In [ ]:
#We now write every anomaly event into the anomaly_events table.

 
from sqlalchemy.orm import Session
 
anomaly_rows = df[df["is_anomaly"] == 1].copy()
print(f"Saving {len(anomaly_rows)} anomaly events to MySQL...")
 
with engine.connect() as conn:
    for _, row in anomaly_rows.iterrows():
 
        #First find the market_data ID for this date + ticker
        result = conn.execute(text("""
            SELECT id FROM market_data
            WHERE ticker = :ticker
              AND DATE(ts) = DATE(:ts)
            LIMIT 1
        """), {"ticker": TICKER, "ts": row["ts"]})
 
        market_row = result.fetchone()
        if market_row is None:
            continue   #Skip if we can't find the matching row
 
        market_id = market_row[0]
 
        #Now insert into anomaly_events
        conn.execute(text("""
            INSERT INTO anomaly_events
                (market_id, iso_score, ensemble_score, severity)
            VALUES
                (:market_id, :iso_score, :ensemble_score, :severity)
            ON DUPLICATE KEY UPDATE iso_score = :iso_score
        """), {
            "market_id":      market_id,
            "iso_score":      float(row["iso_score"]),
            "ensemble_score": float(row["iso_score"]),   # placeholder for now
            "severity":       "HIGH"   if row["iso_score"] >= 0.85
                         else "MEDIUM" if row["iso_score"] >= 0.70
                         else "LOW",
        })
 
    conn.commit()
 
print("Saved to MySQL!")

#### VERIFY WHAT'S IN SQL

In [ ]:
#This query joins anomaly_events with market_data so we can
#see the date and ticker alongside the scores.
 
check = pd.read_sql("""
    SELECT
        md.ticker,
        DATE(md.ts)       AS date,
        md.close,
        ae.iso_score,
        ae.severity
    FROM anomaly_events ae
    JOIN market_data md ON md.id = ae.market_id
    ORDER BY ae.iso_score DESC
    LIMIT 10
""", engine)
 
print("\nTop 10 anomalies stored in MySQL:")
print(check.to_string(index=False))

#### SAVE THE MODEL

In [ ]:
import joblib

joblib.dump(
    {"model": model, "scaler": scaler, "feature_cols": FEATURE_COLS},
    f"isolation_forest_{TICKER}.pk1"
)

print(f"Model saved as isolation_forest_{TICKER}.pk1")
print("")
print("=" * 60)
print("Step 4 complete!")
print("=" * 60)
print(f"  Ticker:            {TICKER}")
print(f"  Training days:     {len(df)}")
print(f"  Anomalies found:   {df['is_anomaly'].sum()}")
print(f"  Anomaly_rate:      {df['is_anomaly'].mean():.1%}")
print(f"  Charts saved:      3 PNG files")
print(f"  Model saved:       isolation_forest_{TICKER}.pk1")
print(f"  MySQL updated:     Anomaly_events table")
print("")
print("Ready for Step 5 -> LSTM Autoenocoder!")

## LSTM AUTOENCODER (SECOND ML MODEL - DEEP LEARNING)

In [ ]:
import sys
!{sys.executable} -m pip install tensorflow scikit-learn joblib

#### IMPORTING TOOLS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import RobustScaler
import joblib
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print("Tensorflow version:", tf.__version__)
print("All tools imported!")

#### CONNECT TO MYSQL

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/Users/Aarushi/.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected! (password hidden safely)")

#### LOAD DATA FROM MYSQL

In [ ]:
TICKER = "AAPL"
 
query = f"""
    SELECT ts, open, high, low, close, volume
    FROM market_data
    WHERE ticker = '{TICKER}'
    ORDER BY ts ASC
"""
 
df = pd.read_sql(query, engine, parse_dates=["ts"])
 
print(f"Loaded {len(df)} rows for {TICKER}")
print(df.head())

#### COMPUTE FEATURES

In [ ]:
def compute_features(df):
    df = df.copy().sort_values("ts").reset_index(drop=True)
 
    #Log return: how much did price change today vs yesterday?
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
 
    #RSI: is the stock overbought or oversold?
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).ewm(span=14, min_periods=14).mean()
    loss  = (-delta).clip(lower=0).ewm(span=14, min_periods=14).mean()
    rs    = gain / (loss + 1e-9)
    df["rsi_14"] = 100 - (100 / (1 + rs))
 
    #Bollinger Band distances: how far is price from normal range?
    ma  = df["close"].rolling(20).mean()
    std = df["close"].rolling(20).std()
    bb_upper = ma + 2 * std
    bb_lower = ma - 2 * std
    df["bb_upper_dist"] = (bb_upper - df["close"]) / (df["close"] + 1e-9)
    df["bb_lower_dist"] = (df["close"] - bb_lower) / (df["close"] + 1e-9)
 
    #VWAP distance: how far is price from "fair value"?
    typical_price = (df["high"] + df["low"] + df["close"]) / 3
    cumtp  = (typical_price * df["volume"]).cumsum()
    cumvol = df["volume"].cumsum()
    vwap   = cumtp / (cumvol + 1e-9)
    df["vwap_dist"] = (df["close"] - vwap) / (vwap + 1e-9)
 
    #Volume ratio: is today's volume unusually high or low?
    df["vol_ratio"] = df["volume"] / (df["volume"].rolling(20).mean() + 1e-9)
 
    df = df.dropna().reset_index(drop=True)
    return df
 
FEATURE_COLS = [
    "log_return", "rsi_14", "bb_upper_dist",
    "bb_lower_dist", "vwap_dist", "vol_ratio"
]
 
df = compute_features(df)
print(f"Features computed! Shape: {df.shape}")

#### SCALING FEATURES

In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS]).astype(np.float32)
 
print(f"Scaled data shape: {X_scaled.shape}")
print("(rows = trading days, columns = 6 features)")

#### BUILD SEQUENCES

In [ ]:
#THIS IS THE KEY DIFFERENCE FROM ISOLATION FOREST

#Instead of feeding the model one day at a time, we feed it WINDOWS of 30 days at a time.

TIMESTEPS = 30

def make_sequences(data, timesteps):
    """
    Slide a window of length `timesteps` over the data.
    Returns an array of shape (n_windows, timesteps, n_features)
    """
    sequences = []
    for i in range(len(data) - timesteps + 1):
        window = data[i : i + timesteps]
        sequences.append(window)
    return np.array(sequences, dtype=np.float32)

X_seq = make_sequences(X_scaled, TIMESTEPS)

print(f"Sequence array shape: {X_seq.shape}")
print(f"  -> {X_seq.shape[0]} windows")
print(f"  -> each window = {X_seq.shape[1]} days * {X_seq.shape[2]} features")

#### SPLIT INTO TRAINING AND TEST SETS

In [ ]:
#We split the data into two parts:
#Training set (85%) = Model learns from this
#Test set (15%) = We check performance on this (Model never sees this during training)

#IMPORTANT : We do not shuffle! Stock data is time_ordered.
#Shuffling would be like reading a book in random order from which model wouldn't learn the time patterns correctly.

#Training = older data (first 85%)
#Test = newer data (last 15%)

split_idx = int(len(X_seq) * 0.85)

X_train = X_seq[:split_idx]
X_test = X_seq[split_idx:]

print(f"Training windows: {len(X_train)}")
print(f"Test windows: {len(X_test)}")
print("")
print("Think of it like:")
print(f"  Training = stock data from {df['ts'].iloc[0].date()} to {df['ts'].iloc[split_idx].date()}")
print(f"  Test     = stock data from {df['ts'].iloc[split_idx].date()} to {df['ts'].iloc[-1].date()}")

#### BUILD THE LSTM AUTOENCODER MODEL

In [ ]:
#Here we actually build the neural network architecture

#It has 3 parts :
#ENCODER: Reads the 30-day sequence and squishes it into a tiny "summary" (like writing notes)
#BOTTLENECK: The tiny summary forces the model to learn only the most important patterns
#DECODER: Takes the tiny summary and tries to reconstruct the originla 30-day sequence

#If the sequence was NORMAL then sequence was accurate.
#If the sequence was WEIRD then reconstruction has big errors.

N_FEATURES = len(FEATURE_COLS)
LATENT_DIM = 16

def build_model (timesteps, n_features, latent_dim):
    inp = layers.Input(shape = (timesteps, n_features), name = "input")
    
    #--ENCODER--
    x = layers.LSTM(64, return_sequences=True, name = "encoder_lstm1")(inp)
    #Dropout: Randomly switches of 20% of neurons during training
    #This prevents the model from "memorising" instead of "learning" like studying without using the same example twice.
    x=layers.Dropout(0.2, name = "encoder_dropout")(x)
    x=layers.LSTM(latent_dim, return_sequences=False, name="encoder_lstm2")(x)
    
    #--BOTTLENECK--
    #RepeatVector copies the summary 30 times (one for each timestep) so the decoder has something to work with.
    x=layers.RepeatVector(timesteps, name="bottleneck")(x)

    #--DECODER--
    #Mirror image of the ENCODER, tries to rebuild the sequence
    x=layers.LSTM(latent_dim, return_sequences=True, name = "decoder_lstm1")(x)
    x=layers.LSTM(64, return_sequences=True, name = "decoder_lstm2")(x)
    #Final layer: outputs the reconstructed sequence
    #TimeDistributed applies a dense layer to each of the 30 timesteps
    output = layers.TimeDistributed(
        layers.Dense(n_features), name="reconstruction")(x)
    
    model = Model(inputs=inp, outputs=output, name="LSTM_Autoencoder")

    #Compile: Tell the model how to learn
    #loss="mse", means: measure error as (predicted-actual)^2
    #optimizer="adam", means: use the ADAM algorithm to improve weights
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse",metrics=["mae"])
    
    return model

model=build_model(TIMESTEPS, N_FEATURES, LATENT_DIM)

model.summary()
print("\nModel built successfully!")

#### TRAIN THE MODEL

In [ ]:
#Teaching the model

#This is an auto-encoder so:
#Input = X_train (the 30-day windows)
#Target = X_train (we want it to reconstruct itself)

#The model will train for upto 50 epochs.
#One epoch = The model looks at ALL training windows once.

#3 helpers (callbacks) watching training:

#EarlyStopping: If the model stops improving for 8 epochs in a row - stop training automatically (prevents wasting time).
#ReduceLROnPlateau: If stuck, reduce the learning rate.
#ModelCheckPoint: Saves the best version of the model automatically during training.

print("Starting training... (this may take 3-10 minutes)")
print("You'll see the loss going down — that means it's learning!")
print("")
 
cb_early_stop = callbacks.EarlyStopping(
    monitor="val_loss",        # watch the validation loss
    patience=8,                # stop if no improvement for 8 epochs
    restore_best_weights=True, # go back to the best version when done
    verbose=1
)
 
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,      # cut learning rate in half when stuck
    patience=4,      # wait 4 epochs before reducing
    min_lr=1e-6,     # don't go below this learning rate
    verbose=1
)
 
cb_checkpoint = callbacks.ModelCheckpoint(
    filepath=f"lstm_{TICKER}_best.h5",  # save file name
    monitor="val_loss",
    save_best_only=True,   # only save when it improves
    verbose=0
)
 
# TRAIN!
# Notice: model.fit(X_train, X_train) — input AND target are the same
# That's what makes it an autoencoder
history = model.fit(
    X_train, X_train,                      # input = target
    validation_data=(X_test, X_test),        # check on validation set
    epochs=50,                             # maximum 50 rounds
    batch_size=64,                         # process 64 windows at a time
    callbacks=[cb_early_stop, cb_reduce_lr, cb_checkpoint],
    verbose=1                              # show progress
)
 
print("\nTraining complete!")
print(f"Best validation loss: {min(history.history['val_loss']):.6f}")

#### PLOT TRAINING HISTORY

In [ ]:
#This chart shows how the model improved over time

#Both lines going down over time (loss decreasing = learning)
#Training and validation lines staying close together
#If they separate, the model is "overfitting" = memorising

fig, axes = plt.subplots(1,2, figsize = (12,4))
fig.suptitle(f"{TICKER} - LSTM Training History", fontsize = 13)

#Left chart: MSE Loss
axes[0].plot(history.history["loss"], color = "#378ADD", label = "Training Loss")
axes[0].plot(history.history["val_loss"], color = "#D85A30", label = "Validation Loss")
axes[0].set_title("Loss (MSE) - lower is better")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

#Right chart: MAE
axes[1].plot(history.history["mae"], color = "#378ADD", label = "MAE")
axes[1].plot(history.history["val_mae"], color = "#D85A30", label = "Validation MAE")
axes[1].set_title("MAE - lower is better")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{TICKER}_lstm_training.png", dpi = 150, bbox_inches = "tight")
plt.show()
print(f"Chart saved as {TICKER}_lstm_training.png")

#### CALCULATE RECONSTRUCTION ERRORS

In [ ]:
#Now we ask the model: "Reconstruct every 30-day window."
#Then we measure: "how wrong were you?"

#The error for each window = average of (predicted - actual)^2
#This is called "Mean Squared Error (MSE)."
#Normal window -> Model reconstructs well -> small error
#Anomaly window -> Model struggles -> large error

#We then set a THRESHOLD: 
#Any window with error above the threshold = flagged as an anomaly.

#We use the 95th percentile of training errors as threshold.
#That means: only the top 5% most unusual windows get flagged.

print("Calculating reconstruction errors...")

#Get model's prediction on ALL data"
X_pred = model.predict(X_seq, verbose=0, batch_size=256)

#Calculate error for each window: mean of (predicted-actual)^2
#axis=(1,2) means average across both the time and feature dimensions
reconstruction_errors = np.mean(np.square(X_seq - X_pred), axis = (1,2))

#Set threshold at 95th percentile of TRAINING errors only
train_errors = reconstruction_errors[:split_idx]
THRESHOLD = np.percentile(train_errors, 95)

print(f"Reconstruction errors stats:")
print(f"  Minimum error:  {reconstruction_errors.min():.6f}")
print(f"  Average error:  {reconstruction_errors.mean():.6f}")
print(f"  Maximum error:  {reconstruction_errors.max():.6f}")
print(f"  THRESHOLD (p95): {THRESHOLD:.6f}")
print("")
print(f"Anything above {THRESHOLD:.6f} will be flagged as an anomaly")

#### FLAG ANOMALIES

In [ ]:
#We add the reconstruction errors back to our dataframe.

#Note: the first (TIMESTEPS-1) = 29 rows get NaN
#because there aren't enough previous days to make a full window.
#Day 1 can't have a 30-day window — it only has 1 day of history!
 
#Create an array of NaN the same length as our dataframe
error_series = np.full(len(df), np.nan)
 
#Fill in errors starting from row 29 (index TIMESTEPS-1)
error_series[TIMESTEPS - 1:] = reconstruction_errors
 
#Add to dataframe
df["lstm_error"] = error_series
df["lstm_anomaly"] = (df["lstm_error"] > THRESHOLD).astype(int)
 
#Show results
total_flagged = df["lstm_anomaly"].sum()
print(f"Total trading days:     {len(df)}")
print(f"Days flagged:           {total_flagged}")
print(f"Anomaly rate:           {df['lstm_anomaly'].mean():.1%}")
 
#Show the top anomalies
top = (
    df[df["lstm_anomaly"] == 1]
    .sort_values("lstm_error", ascending=False)
    [["ts", "close", "log_return", "vol_ratio", "lstm_error"]]
    .head(15)
    .reset_index(drop=True)
)
print(f"\nTop 15 anomalous windows detected by LSTM:")
print("=" * 70)
print(top.to_string(index=False))
print("")
print("TIP: Look up these dates — many will match real market events!")

#### CHART 1: PRICE WITH ANOMALIES HIGHLIGHTED

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize = (14,8), sharex = True)
fig.suptitle(f"{TICKER} - LSTM Autoencoder Anomaly Detection", fontsize=14)

#Top: Price line + Anomaly dots
ax1.plot(df["ts"], df["close"], color = "#378ADD", linewidth = 1, label = "Close price")
mask = df["lstm_anomaly"] == 1
ax1.scatter(
    df.loc[mask, "ts"], df.loc[mask, "close"],
    color = "#D85A30", s = 40, zorder = 5, label = "LSTM Anomaly")
ax1.set_ylabel("Price ($)")
ax1.legend(loc = "upper left")
ax1.set_title("stock price with LSTM detected anomalies")

#Bottom: Reconstruction error over time
ax2.fill_between(df["ts"], df["lstm_error"].fillna(0), alpha = 0.3, color = "#534AB7")
ax2.plot(df["ts"], df["lstm_error"], color = "#534AB7", linewidth = 0.8)
ax2.axhline(y=THRESHOLD, color = "#D85A30", linestyle = "--", linewidth = 1.5, label =f"Threshold = {THRESHOLD:.4f}")
ax2.set_ylabel("Reconstruction Error")
ax2.set_xlabel("Date")
ax2.legend(loc="upper left")
ax2.set_title("Reconstruction error (above red line = anomaly)")

plt.tight_layout()
plt.savefig(f"{TICKER}_lstm_anomalies.png", dpi = 150, bbox_inches = "tight")
plt.show()
print(f"Saved: {TICKER}_lstm_anomalies.png")

#### CHART 2: ERROR DISTRIBUTION

In [ ]:
fig, ax = plt.subplots(figsize = (10,4))

normal_errors = df.loc[df["lstm_anomaly"] == 0, "lstm_error"].dropna()
anomaly_errors = df.loc[df["lstm_anomaly"] == 1, "lstm_error"].dropna()

ax.hist(normal_errors, bins = 50, color = "#378ADD", alpha = 0.7,
        label = "Normal days", density = True)
ax.hist(anomaly_errors, bins = 20, color = "#D85A30", alpha = 0.7,
        label = "Anomalous days", density = True)
ax.axvline(x = THRESHOLD, color = "black", linestyle = "--", linewidth = 1.5,
           label = f"Threshold ({THRESHOLD:.4f})")

ax.set_xlabel("Reconstruction Error (lower = model understood the pattern)")
ax.set_ylabel("Density")
ax.set_title(f"{TICKER} - LSTM Reconstruction Error Distribution")
ax.legend()

plt.tight_layout()
plt.savefig(f"{TICKER}_lstm_distribution.png", dpi = 150, bbox_inches = "tight")
plt.show()

#### COMPARE LSTM VS ISOLATION FOREST RESULTS

In [ ]:
#Let's see how many dates both models agree on.
#When two completely different models flag the same day, that's a very strong signal 
#- that two independent witnesses reporting the same crime.

iso_results = pd.read_sql(f"""
    SELECT DATE(md.ts) as date, ae.iso_score
    FROM anomaly_events ae
    JOIN market_data md ON md.id = ae.market_id
    WHERE md.ticker = '{TICKER}'
      AND ae.iso_score IS NOT NULL
""", engine)

iso_dates = set(iso_results["date"].astype(str))
lstm_dates = set(df[df["lstm_anomaly"] == 1]["ts"].dt.date.astype(str))

both = iso_dates & lstm_dates         #Intersection = Flagged by both
either = iso_dates | lstm_dates       #Union = Flagged by atleast one

print("=" * 50)
print(f" Isolation Forest flagged: {len(iso_dates)} days")
print(f" LSTM flagged:             {len(lstm_dates)} days")
print(f" Both agreed on:           {len(both)} days <- strongest signals!")

if both:
    print("\nDates flagged by BOTH models (most suspicious):")
    for d in sorted (both):
        print(f"   {d}")

#### UPDATE MYSQL WITH LSTM SCORES

In [ ]:
#We update the anomaly_events table with the LSTM error scores.
#For dates flagged by BOTH models, we mark them as HIGH severity.
#For dates flagged only by LSTM, we insert a new row.
 
print(f"\nUpdating MySQL with LSTM results...")
 
lstm_anomaly_rows = df[df["lstm_anomaly"] == 1].copy()
updated = 0
inserted = 0
 
with engine.connect() as conn:
    for _, row in lstm_anomaly_rows.iterrows():
 
        #Find the market_data id for this date
        result = conn.execute(text("""
            SELECT id FROM market_data
            WHERE ticker = :ticker
              AND DATE(ts) = DATE(:ts)
            LIMIT 1
        """), {"ticker": TICKER, "ts": row["ts"]})
 
        market_row = result.fetchone()
        if market_row is None:
            continue
 
        market_id = market_row[0]
 
        #Check if a row already exists (from Isolation Forest)
        existing = conn.execute(text("""
            SELECT id, iso_score FROM anomaly_events
            WHERE market_id = :market_id
            LIMIT 1
        """), {"market_id": market_id}).fetchone()
 
        if existing:
            #UPDATE: add lstm score to existing row
            #Ensemble score = 40% iso + 60% lstm (lstm weighted higher)
            iso_score      = existing[1] or 0.0
            lstm_norm      = float(row["lstm_error"]) / (reconstruction_errors.max() + 1e-9)
            ensemble_score = 0.4 * iso_score + 0.6 * lstm_norm
            severity = "HIGH"   if ensemble_score >= 0.75 \
                  else "MEDIUM" if ensemble_score >= 0.55 \
                  else "LOW"
 
            conn.execute(text("""
                UPDATE anomaly_events
                SET lstm_recon_err  = :lstm,
                    ensemble_score  = :ens,
                    severity        = :sev
                WHERE market_id = :mid
            """), {
                "lstm": float(row["lstm_error"]),
                "ens":  ensemble_score,
                "sev":  severity,
                "mid":  market_id
            })
            updated += 1
        else:
            #INSERT: new anomaly found only by LSTM
            lstm_norm      = float(row["lstm_error"]) / (reconstruction_errors.max() + 1e-9)
            ensemble_score = 0.6 * lstm_norm
            severity = "HIGH"   if ensemble_score >= 0.75 \
                  else "MEDIUM" if ensemble_score >= 0.55 \
                  else "LOW"
 
            conn.execute(text("""
                INSERT INTO anomaly_events
                    (market_id, lstm_recon_err, ensemble_score, severity)
                VALUES (:mid, :lstm, :ens, :sev)
            """), {
                "mid":  market_id,
                "lstm": float(row["lstm_error"]),
                "ens":  ensemble_score,
                "sev":  severity
            })
            inserted += 1
 
    conn.commit()
 
print(f"Updated {updated} existing rows (found by both models)")
print(f"Inserted {inserted} new rows (found only by LSTM)")

#### VERIFY FINAL RESULTS IN MYSQL

In [ ]:
final = pd.read_sql(f"""
    SELECT
        md.ticker,
        DATE(md.ts)      AS date,
        md.close,
        ae.iso_score,
        ae.lstm_recon_err,
        ae.ensemble_score,
        ae.severity
    FROM anomaly_events ae
    JOIN market_data md ON md.id = ae.market_id
    WHERE md.ticker = '{TICKER}'
    ORDER BY ae.ensemble_score DESC
    LIMIT 15
""", engine)
 
print("\nTop 15 anomalies in MySQL (both models combined):")
print("=" * 80)
print(final.to_string(index=False))

#### SAVE EVERYTHING

In [ ]:
#Save the LSTM model, scaler and threshold to disk.
#Combine with Isolation Forest.
 
#Save keras model
model.save(f"lstm_{TICKER}.h5")
 
#Save scaler and threshold together
joblib.dump({
    "scaler":    scaler,
    "threshold": THRESHOLD,
    "features":  FEATURE_COLS,
    "timesteps": TIMESTEPS,
}, f"lstm_{TICKER}_meta.pkl")
 
print(f"Saved: lstm_{TICKER}.h5")
print(f"Saved: lstm_{TICKER}_meta.pkl")
print("")
print("=" * 60)
print("STEP 5 COMPLETE!")
print("=" * 60)
print(f"  Ticker:              {TICKER}")
print(f"  Training windows:    {len(X_train)}")
print(f"  Anomaly threshold:   {THRESHOLD:.6f}")
print(f"  LSTM anomalies:      {df['lstm_anomaly'].sum()}")
print(f"  Charts saved:        3 PNG files")
print(f"  Model saved:         lstm_{TICKER}.h5")
print(f"  Meta saved:          lstm_{TICKER}_meta.pkl")
print(f"  MySQL updated:       anomaly_events table")
print("")
print("Ready for Step 6 → Combine both models + final results!")

In [ ]:
#Re-save the model in the new Keras format (no .h5 extension)
model.save(f"lstm_{TICKER}_v2.keras")
print("Model re-saved in new format!")

### FINAL ENSEMBLE + SUMMARY DASHBOARD

In [ ]:
#We now have TWO trained models:
#- Isolation Forest : good at spotting one-day oddities
#- LSTM Autoencoder : good at spotting pattern breaks

#Think of them like two detectives:
#Detective 1 (Isolation Forest): "This single day looks very weird"
#Detective 2 (LSTM):             "The last 30 days feel off"

#Now both detectives at the same table and we ask:
#"What do you BOTH agree is suspicious?"

#When two completely different methods flag the same event,
#we can be much more confident it's a real anomaly.

#This final combined score is called an ENSEMBLE score.
#Ensemble = a group working together (like an orchestra).

#IMPORTING TOOLS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sqlalchemy import create_engine, text
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
import joblib
import os
import warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print("All tools imported!")

#### CONNECT TO MYSQL

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/Users/Aarushi/.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected! (password hidden safely)")

#### LOAD SAVED MODELS FROM DISK

In [ ]:
#We saved our trained models and now we load them back just like loading back your saved game
#We saved:
#isolation_forest_AAPL.pk1 = Isolation Forest model + scaler
#lstm_AAPL.h5 = LSTM neural network weights
#lstm_AAPL_meta.pk1 = LSTM scaler + threshold

import os
os.chdir("/Users/Aarushi")  # tells Jupyter to look in the right folder
print("Now looking in:", os.getcwd())

print("Loading saved models...")

#Load Isolation Forest
iso_data     = joblib.load("isolation_forest_AAPL.pk1")
iso_model    = iso_data["model"]
iso_scaler   = iso_data["scaler"]
FEATURE_COLS = iso_data["feature_cols"]
print("   Isolation Forest loaded")

#Load LSTM — using new .keras format
lstm_model = tf.keras.models.load_model("lstm_AAPL_v2.keras")
lstm_meta  = joblib.load("lstm_AAPL_meta.pkl")
lstm_scaler    = lstm_meta["scaler"]
lstm_threshold = lstm_meta["threshold"]
TIMESTEPS      = lstm_meta["timesteps"]

print(f"   LSTM Autoencoder loaded")
print(f"   LSTM threshold: {lstm_threshold:.6f}")
print(f"   Features: {FEATURE_COLS}")

#### LOAD AND PREPARE DATA

In [ ]:
query = f"""
    SELECT ts, open, high, low, close, volume
    FROM market_data
    WHERE ticker = '{TICKER}'
    ORDER BY ts ASC
"""
df_raw = pd.read_sql(query, engine, parse_dates=["ts"])
print(f"Loaded {len(df_raw)} rows from MySQL")
 
#Compute features
def compute_features(df):
    df = df.copy().sort_values("ts").reset_index(drop=True)
    df["log_return"]    = np.log(df["close"] / df["close"].shift(1))
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).ewm(span=14, min_periods=14).mean()
    loss  = (-delta).clip(lower=0).ewm(span=14, min_periods=14).mean()
    df["rsi_14"]        = 100 - (100 / (1 + gain / (loss + 1e-9)))
    ma  = df["close"].rolling(20).mean()
    std = df["close"].rolling(20).std()
    df["bb_upper_dist"] = (ma + 2*std - df["close"]) / (df["close"] + 1e-9)
    df["bb_lower_dist"] = (df["close"] - (ma - 2*std)) / (df["close"] + 1e-9)
    typical = (df["high"] + df["low"] + df["close"]) / 3
    vwap    = (typical * df["volume"]).cumsum() / (df["volume"].cumsum() + 1e-9)
    df["vwap_dist"]     = (df["close"] - vwap) / (vwap + 1e-9)
    df["vol_ratio"]     = df["volume"] / (df["volume"].rolling(20).mean() + 1e-9)
    return df.dropna().reset_index(drop=True)
 
df = compute_features(df_raw)
print(f"Features computed. Shape: {df.shape}")

#### GET ISOLATION FOREST SCORES

In [ ]:
#Run the Isolation Forest model on all data.
#Returns a score between 0 and 1 for every day.
#Higher = more anomalous

print("Running Isolation Forest...")

#Re-scale using iso scaler
X_iso = iso_scaler.transform(df[FEATURE_COLS])

#Get raw scores and normalise to 0-1
raw        = iso_model.score_samples(X_iso)
iso_scores = (-raw - (-raw).min()) / ((-raw).max() - (-raw).min())

#Add to dataframe
df["iso_score"] = iso_scores

#Verify it worked
print(f"  ✓ iso_score column added: {df['iso_score'].shape[0]} values")
print(f"  Score range: {iso_scores.min():.4f} to {iso_scores.max():.4f}")
print(df[["ts", "close", "iso_score"]].tail(5))  # shows last 5 rows as proof

#### GET LSTM RECONSTRUCTION ERROR SCORES

In [ ]:
#Run the LSTM model on all data.
#Returns a reconstruction error for each 30-day window.
#Higher error = more anomalous

print("Running LSTM Autoencoder...")
 
# Scale using LSTM's own scaler
X_lstm = lstm_scaler.transform(df[FEATURE_COLS]).astype(np.float32)
 
#Build 30-day windows
def make_sequences(data, timesteps):
    return np.array([data[i:i+timesteps] for i in range(len(data)-timesteps+1)],
                    dtype=np.float32)
 
X_seq  = make_sequences(X_lstm, TIMESTEPS)
X_pred = lstm_model.predict(X_seq, verbose=0, batch_size=256)
 
#Reconstruction error per window
lstm_errors = np.mean(np.square(X_seq - X_pred), axis=(1, 2))
 
#Pad first 29 rows with NaN (no full window available)
lstm_error_series = np.full(len(df), np.nan)
lstm_error_series[TIMESTEPS - 1:] = lstm_errors
 
#Normalise to 0–1 range (same scale as iso_score)
valid_errors = lstm_errors
lstm_norm = (valid_errors - valid_errors.min()) / \
            (valid_errors.max() - valid_errors.min() + 1e-9)
 
lstm_norm_series = np.full(len(df), np.nan)
lstm_norm_series[TIMESTEPS - 1:] = lstm_norm
 
df["lstm_error"]      = lstm_error_series
df["lstm_score_norm"] = lstm_norm_series
 
print(f"  LSTM complete.")
print(f"  Error range: {lstm_errors.min():.6f} to {lstm_errors.max():.6f}")

#### COMBINE INTO ONE ENSEMBLE SCORE

In [ ]:
#This is the heart of the whole project
#We combine both scores with weights:
#- 40% Isolation Forest - (fast, good at single-day oddities)
#- 60% LSTM             - (deeper, good at pattern breaks)

#LSTM gets slightly more weight because it uses context (30 days) vs Isolation Forest (1 day at a time).
#For rows where LSTM score is NaN (first 29 days), we fall back to just the Isolation score.

ISO_WEIGHT  = 0.4
LSTM_WEIGHT = 0.6
 
# Where both scores exist → weighted average
both_available = ~df["lstm_score_norm"].isna()
df.loc[both_available, "ensemble_score"] = (
    ISO_WEIGHT  * df.loc[both_available, "iso_score"] +
    LSTM_WEIGHT * df.loc[both_available, "lstm_score_norm"]
)
 
#Where only ISO exists (first 29 rows) : use ISO only
df.loc[~both_available, "ensemble_score"] = df.loc[~both_available, "iso_score"]
 
#Assign severity labels
#Think of it like an alert system:
#HIGH   = call the trading desk NOW
#MEDIUM = keep a close eye on this
#LOW    = worth noting, not urgent
def assign_severity(score):
    if score >= 0.75:  return "HIGH"
    if score >= 0.55:  return "MEDIUM"
    if score >= 0.40:  return "LOW"
    return "NORMAL"
 
df["severity"]    = df["ensemble_score"].apply(assign_severity)
df["is_anomaly"]  = (df["severity"] != "NORMAL").astype(int)
 
#Summary
print("\nEnsemble scoring complete!")
print(f"  NORMAL:  {(df['severity']=='NORMAL').sum():>5} days")
print(f"  LOW:     {(df['severity']=='LOW').sum():>5} days")
print(f"  MEDIUM:  {(df['severity']=='MEDIUM').sum():>5} days")
print(f"  HIGH:    {(df['severity']=='HIGH').sum():>5} days")

#### PRINT THE FINAL ANOMALY REPORT

In [ ]:
#A clean table of the most significant anomaly events sorted by ensemble score.

top_anomalies = (
    df[df["severity"].isin(["HIGH", "MEDIUM"])]
    .sort_values("ensemble_score", ascending=False)
    [[
        "ts", "close", "log_return",
        "vol_ratio", "iso_score", 
        "lstm_score_norm", "ensemble_score", "severity"
    ]]
    .head(20)
    .reset_index(drop=True)
)

#Round for readability
top_anomalies["close"]            = top_anomalies["close"].round(2)
top_anomalies["log_return"]       = top_anomalies["log_return"].round(4)
top_anomalies["vol_Ratio"]        = top_anomalies["vol_ratio"].round(2)
top_anomalies["iso_score"]        = top_anomalies["iso_score"].round(4)
top_anomalies["lstm_score_norm"]  = top_anomalies["lstm_score_norm"].round(4)
top_anomalies["ensemble_score"]   = top_anomalies["ensemble_score"].round(4)
top_anomalies["ts"]               = top_anomalies["ts"].dt.strftime("%Y-%m-%d")

print("\n" + "=" * 90)
print(f" FINAL ANOMALY REPORT - {TICKER}")
print("=" * 90)
print(top_anomalies.to_string(index=False))
print("=" * 90)

#### THE BIG DASHBOARD CHART

In [ ]:
#This is my showpiece visualisation.
#4 panels stacked vertically, all sharing the same time axis:

#Panel 1: Stock price with HIGH anomalies marked as red dots.
#Panel 2: Isolation Forest score over time.
#Panel 3: LSTM reconstruction error over time.
#Panel 4: Final ensemble score with severity color bands.

fig = plt.figure(figsize=(16, 14))
fig.suptitle(f"{TICKER} — Real-Time Market Anomaly Detection System\n"
             f"Isolation Forest (40%) + LSTM Autoencoder (60%) Ensemble",
             fontsize=14, fontweight="bold", y=0.98)
 
gs = gridspec.GridSpec(4, 1, figure=fig, hspace=0.45)
 
#Colour palette
COL_PRICE  = "#2C7BB6"
COL_ISO    = "#1D9E75"
COL_LSTM   = "#534AB7"
COL_ENS    = "#D85A30"
COL_HIGH   = "#D32F2F"
COL_MED    = "#F57C00"
COL_LOW    = "#FBC02D"
COL_SHADE  = "#FDECEA"
 
#── Panel 1: Price ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(df["ts"], df["close"], color=COL_PRICE, linewidth=1.2, label="Close price")
 
#Mark HIGH anomalies as red dots
high_mask = df["severity"] == "HIGH"
med_mask  = df["severity"] == "MEDIUM"
ax1.scatter(df.loc[high_mask, "ts"], df.loc[high_mask, "close"],
            color=COL_HIGH, s=50, zorder=5, label="HIGH anomaly")
ax1.scatter(df.loc[med_mask, "ts"],  df.loc[med_mask, "close"],
            color=COL_MED,  s=25, zorder=4, label="MEDIUM anomaly", alpha=0.7)
 
ax1.set_ylabel("Price ($)", fontsize=10)
ax1.set_title("Stock Price with Anomaly Alerts", fontsize=11)
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(axis="y", alpha=0.3)
 
#── Panel 2: Isolation Forest score ──────────────────────────
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.fill_between(df["ts"], df["iso_score"], alpha=0.25, color=COL_ISO)
ax2.plot(df["ts"], df["iso_score"], color=COL_ISO, linewidth=0.8)
ax2.axhline(y=0.65, color=COL_HIGH, linestyle="--", linewidth=1,
            label="Alert threshold (0.65)")
ax2.set_ylabel("Score (0–1)", fontsize=10)
ax2.set_title("Isolation Forest Score — single-day anomalies", fontsize=11)
ax2.legend(loc="upper left", fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.grid(axis="y", alpha=0.3)
 
#── Panel 3: LSTM score ───────────────────────────────────────
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.fill_between(df["ts"], df["lstm_score_norm"].fillna(0),
                 alpha=0.25, color=COL_LSTM)
ax3.plot(df["ts"], df["lstm_score_norm"], color=COL_LSTM, linewidth=0.8)
lstm_thresh_norm = (lstm_threshold - lstm_errors.min()) / \
                   (lstm_errors.max() - lstm_errors.min() + 1e-9)
ax3.axhline(y=lstm_thresh_norm, color=COL_HIGH, linestyle="--", linewidth=1,
            label=f"Alert threshold ({lstm_thresh_norm:.2f})")
ax3.set_ylabel("Score (0–1)", fontsize=10)
ax3.set_title("LSTM Autoencoder Score — pattern-break anomalies", fontsize=11)
ax3.legend(loc="upper left", fontsize=9)
ax3.set_ylim(0, 1.05)
ax3.grid(axis="y", alpha=0.3)
 
#── Panel 4: Ensemble score ───────────────────────────────────
ax4 = fig.add_subplot(gs[3], sharex=ax1)
 
#Colour background bands for severity zones
ax4.axhspan(0.75, 1.05, alpha=0.08, color=COL_HIGH,  label="HIGH zone")
ax4.axhspan(0.55, 0.75, alpha=0.08, color=COL_MED,   label="MEDIUM zone")
ax4.axhspan(0.40, 0.55, alpha=0.08, color=COL_LOW,   label="LOW zone")
 
ax4.fill_between(df["ts"], df["ensemble_score"], alpha=0.3, color=COL_ENS)
ax4.plot(df["ts"], df["ensemble_score"], color=COL_ENS, linewidth=1)
 
#Threshold lines
ax4.axhline(y=0.75, color=COL_HIGH, linestyle="--", linewidth=0.8, alpha=0.7)
ax4.axhline(y=0.55, color=COL_MED,  linestyle="--", linewidth=0.8, alpha=0.7)
ax4.axhline(y=0.40, color=COL_LOW,  linestyle="--", linewidth=0.8, alpha=0.7)
 
ax4.set_ylabel("Score (0–1)", fontsize=10)
ax4.set_xlabel("Date", fontsize=10)
ax4.set_title("Final Ensemble Score (40% ISO + 60% LSTM) — combined alert",
              fontsize=11)
ax4.legend(loc="upper left", fontsize=9, ncol=3)
ax4.set_ylim(0, 1.05)
ax4.grid(axis="y", alpha=0.3)
 
plt.savefig(f"{TICKER}_final_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Dashboard saved as {TICKER}_final_dashboard.png")

#### STATISTICS SUMMARY

In [ ]:
total_days    = len(df)
high_count    = (df["severity"] == "HIGH").sum()
medium_count  = (df["severity"] == "MEDIUM").sum()
low_count     = (df["severity"] == "LOW").sum()
anomaly_count = df["is_anomaly"].sum()
date_start    = df["ts"].min().strftime("%Y-%m-%d")
date_end      = df["ts"].max().strftime("%Y-%m-%d")
 
#Dates where BOTH models independently flagged an anomaly
iso_flagged   = set(df[df["iso_score"]       > 0.65]["ts"].dt.date.astype(str))
lstm_flagged  = set(df[df["lstm_score_norm"] > lstm_thresh_norm]["ts"].dt.date.astype(str))
both_flagged  = iso_flagged & lstm_flagged
 
print("\n" + "=" * 60)
print("  PROJECT SUMMARY — USE THIS IN YOUR INTERVIEW")
print("=" * 60)
print(f"  Ticker analysed:       {TICKER}")
print(f"  Date range:            {date_start} to {date_end}")
print(f"  Total trading days:    {total_days}")
print(f"  Models used:           Isolation Forest + LSTM Autoencoder")
print(f"  Ensemble weights:      40% ISO + 60% LSTM")
print(f"")
print(f"  ── Anomaly breakdown ──────────────────────────")
print(f"  HIGH severity:         {high_count} days")
print(f"  MEDIUM severity:       {medium_count} days")
print(f"  LOW severity:          {low_count} days")
print(f"  Total flagged:         {anomaly_count} days ({anomaly_count/total_days:.1%})")
print(f"")
print(f"  ── Model agreement ────────────────────────────")
print(f"  ISO Forest flagged:    {len(iso_flagged)} days")
print(f"  LSTM flagged:          {len(lstm_flagged)} days")
print(f"  BOTH agreed on:        {len(both_flagged)} days ← strongest signals")
print("=" * 60)

#### SAVE FINAL RESULTS TO MYSQL

In [ ]:
#Write the final ensemble scores to the anomaly_events table.
#This overwrites/updates what was there before with the proper combined ensemble score.

print("\nSaving final ensemble results to MySQL...")
 
anomaly_df = df[df["is_anomaly"] == 1].copy()
saved = 0
 
with engine.connect() as conn:
    for _, row in anomaly_df.iterrows():
 
        result = conn.execute(text("""
            SELECT id FROM market_data
            WHERE ticker = :ticker AND DATE(ts) = DATE(:ts)
            LIMIT 1
        """), {"ticker": TICKER, "ts": row["ts"]})
        market_row = result.fetchone()
        if market_row is None:
            continue
 
        market_id = market_row[0]
 
        conn.execute(text("""
            INSERT INTO anomaly_events
                (market_id, iso_score, lstm_recon_err,
                 ensemble_score, severity)
            VALUES
                (:mid, :iso, :lstm, :ens, :sev)
            ON DUPLICATE KEY UPDATE
                iso_score      = :iso,
                lstm_recon_err = :lstm,
                ensemble_score = :ens,
                severity       = :sev
        """), {
            "mid":  market_id,
            "iso":  float(row["iso_score"]),
            "lstm": float(row["lstm_error"]) if not pd.isna(row["lstm_error"]) else None,
            "ens":  float(row["ensemble_score"]),
            "sev":  row["severity"],
        })
        saved += 1
 
    conn.commit()
 
print(f"Saved {saved} anomaly events to MySQL.")

#### FINAL VERIFICATION QUERY

In [ ]:
#Pull the final results from MySQL and display them.
#This is what a production system would serve to a dashboard.

final_results = pd.read_sql(f"""
    SELECT 
        md.ticker,
        DATE(md.ts)          AS date,
        md.close             AS price,
        ROUND(ae.iso_score, 4)          AS iso_score,
        ROUND(ae.lstm_recon_err, 6)    AS lstm_error,
        ROUND(ae.ensemble_score, 4)    AS ens,
        ae.severity
    FROM anomaly_events ae
    JOIN market_data md ON md.id = ae.market_id
    WHERE md.ticker = '{TICKER}'
    ORDER BY ae.ensemble_score DESC
    LIMIT 20
""", engine)

print("\nFinal results from MySQL (top 20 ensemble score:)")
print("=" * 80)
print(final_results.to_string(index=False))
print("=" * 80)

#### SAVE THE ENSEMBLE RESULTS TO CSV

In [ ]:
output_df = df[[
    "ts", "close", "log_return", "vol_ratio",
    "iso_score", "lstm_score_norm", "ensemble_score", "severity"]].copy()

output_df.columns = [
    "date", "close_price", "log_return", "volume_ratio",
    "iso_forest_score", "lstm_score", "ensemble_score", "severity"]

output_df["date"] = output_df["date"].dt.strftime("%Y-%m-%d")
output_df = output_df.round(6)

csv_path = f"{TICKER}_anomaly_results_csv"
output_df.to_csv(csv_path, index = False)

print(f"Full results exported to: {csv_path}")
print(f"Total row {len(output_df)}")